
# Auslan Fingerspelling Recognition with an LSTM

Recognises the Auslan (Australian Sign Language) fingerspelling alphabet A-Z from a webcam.

Auslan fingerspelling is two-handed and several letters involve movement, so a single image of a hand
isn't enough to tell letters apart. Instead of classifying still images, this treats each letter as a
short *sequence*: 30 frames of hand keypoints, fed into an LSTM.

The pipeline is:

```
webcam -> MediaPipe Holistic -> 21 keypoints per hand (both hands)
       -> normalise each hand -> 30-frame sliding window -> LSTM -> letter
```

### Data
26 letters x 50 sequences x 30 frames, recorded with MediaPipe Holistic. Each frame stores the
x/y/z coordinates of 21 landmarks on each hand, so 21 * 3 * 2 = 126 numbers per frame.

The original recordings are one .npy file per frame (39,000 files, ~390MB). That's slow to load and
annoying to move around, so everything has been packed into a single `auslan_hand_sequences.npz`
(16MB) with the arrays already stacked. Section 2 loads that, but there's also a loader for the
original folder layout if you want to record more data yourself.

### Contents
1. Setup
2. Load data
3. Look at the data
4. Normalisation
5. Augmentation
6. Train/test split
7. LSTM model
8. Training
9. Evaluation
10. Real-time detection
11. Notes and limitations


## 1. Setup

In [ ]:

# Versions are pinned because the newest versions of these don't play nicely together:
#  - TensorFlow >= 2.18 wants protobuf >= 6, mediapipe 0.10.14 wants protobuf < 5
#  - mediapipe >= 0.10.18 dropped mp.solutions.holistic, which section 10 uses
#  - opencv >= 4.12 wants numpy >= 2
import sys
!{sys.executable} -m pip install --quiet "numpy<2.0" "protobuf==4.25.3"
!{sys.executable} -m pip install --quiet "tensorflow==2.17.1"
!{sys.executable} -m pip install --quiet "opencv-python==4.10.0.84" "mediapipe==0.10.14"
!{sys.executable} -m pip install --quiet scikit-learn scipy matplotlib seaborn pandas


In [ ]:

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

ACTIONS = np.array([chr(c) for c in range(ord('A'), ord('Z') + 1)])
SEQUENCE_LENGTH = 30
NO_SEQUENCES = 50
N_FEATURES = 126          # 21 landmarks * 3 coords * 2 hands

label_map = {label: i for i, label in enumerate(ACTIONS)}

NPZ_PATH = "auslan_hand_sequences.npz"
DATA_PATH = "AUSLAN_Data"      # only needed if loading from the original folders
MODEL_PATH = "best_model.keras"

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "none, running on CPU")



## 2. Load data

Two options here. The `.npz` is the fast path. The folder loader does the same thing but reads the
original one-file-per-frame layout, which is what you'd get if you recorded your own data with
section 10's collection loop.

Note the `[-126:]` slice in the folder loader. MediaPipe Holistic returns 1662 values per frame
(pose + face + both hands). The hands are the last 126, and everything else gets thrown away - face
and pose landmarks are just noise for fingerspelling and would make the input 13x bigger.


In [ ]:

def load_from_npz(path=NPZ_PATH):
    d = np.load(path, allow_pickle=True)
    return d["X"].astype(np.float32), d["y"].astype(int)


def load_from_folders(data_path=DATA_PATH):
    sequences, labels = [], []
    for action in ACTIONS:
        for seq_num in range(NO_SEQUENCES):
            window = []
            for frame_num in range(SEQUENCE_LENGTH):
                f = os.path.join(data_path, action, str(seq_num), f"{frame_num}.npy")
                res = np.load(f)
                window.append(res[-N_FEATURES:])   # keep hands only
            sequences.append(np.array(window, dtype=np.float32))
            labels.append(label_map[action])
    return np.array(sequences, dtype=np.float32), np.array(labels)


if os.path.exists(NPZ_PATH):
    X_raw, y_labels = load_from_npz()
    print("loaded from npz")
elif os.path.isdir(DATA_PATH):
    X_raw, y_labels = load_from_folders()
    print("loaded from folders")
else:
    raise FileNotFoundError(
        f"Couldn't find {NPZ_PATH} or the {DATA_PATH}/ folder. Put the npz next to this notebook."
    )

print("X_raw:", X_raw.shape, "  y:", y_labels.shape)
print("sequences per letter:", np.bincount(y_labels))


## 3. Look at the data

In [ ]:

# How often is each hand actually detected? MediaPipe returns zeros when it can't find a hand,
# so this is worth checking before training on it.
left  = X_raw[:, :, :63]
right = X_raw[:, :, 63:]
left_present  = (np.abs(left).sum(axis=2)  > 0)
right_present = (np.abs(right).sum(axis=2) > 0)

print(f"frames with left hand detected:  {left_present.mean():.1%}")
print(f"frames with right hand detected: {right_present.mean():.1%}")
print(f"frames with both hands:          {(left_present & right_present).mean():.1%}")
print(f"frames with neither:             {(~left_present & ~right_present).mean():.1%}")

per_letter = pd.DataFrame({
    "letter": ACTIONS[y_labels],
    "both_hands": (left_present & right_present).mean(axis=1),
})
plt.figure(figsize=(12, 4))
per_letter.groupby("letter")["both_hands"].mean().reindex(ACTIONS).plot(kind="bar")
plt.ylabel("fraction of frames with both hands")
plt.title("Two-hand detection rate per letter")
plt.tight_layout(); plt.show()


In [ ]:

# Plot how one keypoint moves over the 30 frames, for a few letters.
# Letters that involve movement should show a lot more variation than static ones.
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, letter in zip(axes, ["A", "C", "J", "P"]):
    idx = np.where(y_labels == label_map[letter])[0][0]
    seq = X_raw[idx]
    ax.plot(seq[:, 63], label="right wrist x")      # first coord of right hand
    ax.plot(seq[:, 64], label="right wrist y")
    ax.set_title(letter); ax.set_xlabel("frame")
axes[0].set_ylabel("coordinate"); axes[0].legend(fontsize=8)
plt.suptitle("Raw keypoint movement across a sequence")
plt.tight_layout(); plt.show()



## 4. Normalisation

Raw MediaPipe coordinates are relative to the image frame, so the same letter signed closer to the
camera, or off to one side, gives completely different numbers. Without fixing this the model just
learns where you happened to be standing.

For each hand in each frame:
- subtract the wrist (landmark 0), so the wrist sits at the origin
- divide by the distance from wrist to middle finger MCP (landmark 9), so hand size doesn't matter

Missing hands stay as zeros, which the model learns to read as "nothing there".

This function gets used in training *and* in the live webcam loop in section 10. Same function both
times, so there's no chance of the two drifting apart.


In [ ]:

def normalize_keypoints(sequence):
    # sequence: (frames, 126) -> (frames, 126), normalised per hand
    sequence = sequence.reshape(sequence.shape[0], -1, 3)   # (frames, 42, 3)
    out = []
    for frame in sequence:
        hands = []
        for hand in (frame[:21, :], frame[21:, :]):
            if np.any(hand):
                hand = hand - hand[0, :]                     # centre on wrist
                size = np.linalg.norm(hand[9, :])            # wrist -> middle MCP
                hand = hand / size if size > 0 else np.zeros((21, 3))
            else:
                hand = np.zeros((21, 3))
            hands.append(hand)
        out.append(np.concatenate(hands, axis=0).flatten())
    return np.array(out)


X = np.array([normalize_keypoints(seq) for seq in X_raw], dtype=np.float32)
y = to_categorical(y_labels).astype(int)

print("X:", X.shape, " y:", y.shape)
print(f"value range before: {X_raw.min():.2f} to {X_raw.max():.2f}")
print(f"value range after:  {X.min():.2f} to {X.max():.2f}")


In [ ]:

# Sanity check that normalisation actually removed position/scale differences.
# Two sequences of the same letter should look more alike after normalising.
a, b = np.where(y_labels == label_map["B"])[0][:2]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(X_raw[a][:, 63:69]); axes[0].set_title("raw - two 'B' sequences overlap poorly")
axes[0].plot(X_raw[b][:, 63:69], ls="--")
axes[1].plot(X[a][:, 63:69]);     axes[1].set_title("normalised - much closer")
axes[1].plot(X[b][:, 63:69], ls="--")
for ax in axes: ax.set_xlabel("frame")
plt.tight_layout(); plt.show()



## 5. Augmentation

1300 sequences across 26 classes is only 50 examples per letter, which isn't much for an LSTM. Three
augmentations, each applied with 50% probability:

- **noise** - small Gaussian noise, simulates jittery keypoint detection
- **scaling** - multiplies the whole sequence, simulates hand size differences
- **time warping** - stretches/squashes the sequence in time using a cubic spline, simulates signing
  faster or slower

Time warping is the useful one for a sequence model. The other two you'd get from a still-image model
too, but speed variation is specific to sequences and it's a real source of variation between people.


In [ ]:

from scipy.interpolate import CubicSpline

def add_noise(sequence, noise_level=0.05):
    return sequence + np.random.normal(0, noise_level, sequence.shape)

def scale_sequence(sequence, scale_factor=0.2):
    return sequence * np.random.normal(1.0, scale_factor)

def time_warp(sequence, sigma=0.2):
    n = sequence.shape[0]
    warp = np.random.normal(loc=1.0, scale=sigma, size=n)
    cum = np.cumsum(warp)
    cum = (cum - cum.min()) / (cum.max() - cum.min()) * (n - 1)
    cs = CubicSpline(np.arange(n), sequence, axis=0)
    return cs(cum)

def augment_sequence(sequence):
    s = sequence.copy()
    if np.random.rand() < 0.5: s = add_noise(s)
    if np.random.rand() < 0.5: s = scale_sequence(s)
    if np.random.rand() < 0.5: s = time_warp(s)
    return s


In [ ]:

# See what the augmentations do to one sequence
example = X[np.where(y_labels == label_map["D"])[0][0]]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for ax, (name, seq) in zip(axes, [
    ("original", example),
    ("+ noise", add_noise(example)),
    ("+ scaled", scale_sequence(example)),
    ("+ time warp", time_warp(example)),
]):
    ax.plot(seq[:, 63:66]); ax.set_title(name); ax.set_xlabel("frame")
plt.tight_layout(); plt.show()



## 6. Train/test split

Split first, *then* augment the training set only. Augmenting before splitting would put a sequence
and its own noisy copy on opposite sides of the split, which inflates the test score.

One thing worth being upfront about: the split is random over sequences, and all the recordings come
from the same person in the same setup. So the test score measures "can it recognise letters signed
by this person", not "can it recognise letters signed by anyone". See section 11.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y_labels
)

# Augment training data only, doubling its size
aug_X = np.array([augment_sequence(s) for s in X_train], dtype=np.float32)
X_train_aug = np.concatenate([X_train, aug_X], axis=0)
y_train_aug = np.concatenate([y_train, y_train], axis=0)
X_train_aug, y_train_aug = shuffle(X_train_aug, y_train_aug, random_state=42)

print(f"train: {X_train.shape[0]} -> {X_train_aug.shape[0]} after augmentation")
print(f"test:  {X_test.shape[0]}")

# Class weights, in case the split left some letters slightly under-represented
flat = np.argmax(y_train_aug, axis=1)
weights = compute_class_weight('balanced', classes=np.unique(flat), y=flat)
class_weights = dict(enumerate(weights))
print(f"class weight range: {min(weights):.3f} to {max(weights):.3f}")



## 7. Model

Three stacked LSTM layers (64 -> 128 -> 64), then dense layers to 26 outputs. The first two return
sequences so the next LSTM gets the full time series; the last one collapses to a single vector.

BatchNorm + Dropout(0.5) after each LSTM, which matters a lot with only ~2000 training sequences -
without them this overfits within a few epochs.


In [ ]:

model = Sequential([
    LSTM(64, return_sequences=True, activation='relu',
         input_shape=(SEQUENCE_LENGTH, N_FEATURES)),
    BatchNormalization(),
    Dropout(0.5),

    LSTM(128, return_sequences=True, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    LSTM(64, activation='relu'),
    BatchNormalization(),

    Dense(64, activation='relu'),
    Dense(len(ACTIONS), activation='softmax'),
])

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['categorical_accuracy'])
model.summary()


## 8. Training

In [ ]:

callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    X_train_aug, y_train_aug,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    class_weight=class_weights,
)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history['categorical_accuracy'], label='train')
axes[0].plot(history.history['val_categorical_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 9. Evaluation

In [ ]:

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss:     {loss:.4f}")
print(f"Test accuracy: {acc:.4f}")

y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test, axis=1)
confidence = y_prob.max(axis=1)


In [ ]:

print(classification_report(y_true, y_pred, target_names=ACTIONS, zero_division=0))
report = classification_report(y_true, y_pred, target_names=ACTIONS,
                               output_dict=True, zero_division=0)


In [ ]:

df_metrics = pd.DataFrame(report).transpose().iloc[:-3, :]
df_metrics[['precision', 'recall', 'f1-score']].plot(kind='bar', figsize=(14, 5))
plt.title('Precision, recall and F1 per letter')
plt.xlabel('letter'); plt.ylabel('score'); plt.xticks(rotation=0)
plt.legend(loc='lower right'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:

cm = confusion_matrix(y_true, y_pred, labels=range(len(ACTIONS)))
plt.figure(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ACTIONS, yticklabels=ACTIONS, cbar=False, annot_kws={"size": 8})
plt.xlabel('predicted'); plt.ylabel('actual'); plt.title('Confusion matrix')
plt.tight_layout(); plt.show()


In [ ]:

# Which letters get mixed up most
pairs = [(ACTIONS[i], ACTIONS[j], int(cm[i, j]))
         for i in range(len(ACTIONS)) for j in range(len(ACTIONS))
         if i != j and cm[i, j] > 0]
pairs.sort(key=lambda t: -t[2])
if pairs:
    print("Most common confusions:")
    for a, b, n in pairs[:10]:
        print(f"  {a} predicted as {b}: {n}")
else:
    print("No confusions on the test set.")


In [ ]:

# Is the confidence score meaningful? If correct predictions aren't more confident than
# wrong ones, the threshold in the live demo won't do anything useful.
ok, bad = confidence[y_pred == y_true], confidence[y_pred != y_true]
print(f"mean confidence when correct: {ok.mean():.3f}" if len(ok) else "no correct preds")
print(f"mean confidence when wrong:   {bad.mean():.3f}" if len(bad) else "no wrong preds")

bins = np.linspace(0, 1, 31)
plt.figure(figsize=(9, 4))
if len(ok):  plt.hist(ok,  bins=bins, alpha=.7, label='correct')
if len(bad): plt.hist(bad, bins=bins, alpha=.7, label='wrong')
plt.axvline(0.7, ls='--', c='k', label='live threshold (0.7)')
plt.xlabel('confidence'); plt.ylabel('count'); plt.legend()
plt.title('Prediction confidence'); plt.tight_layout(); plt.show()



## 10. Real-time detection

Keeps a rolling buffer of the last 30 frames and predicts on every frame once the buffer is full.
A letter only gets added to the sentence when:

- the last 10 predictions all agree, and
- confidence is above 0.7, and
- it isn't the same as the last letter added (otherwise holding a sign spams it)

The probability bars down both sides are handy for debugging - if a letter is close but losing, you
can see it rather than just getting a wrong answer.

Press `q` to quit.


In [ ]:

import cv2
import mediapipe as mp

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils


def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR), results


def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2))
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))


def extract_keypoints(results):
    lh = (np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten()
          if results.left_hand_landmarks else np.zeros(21 * 3))
    rh = (np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten()
          if results.right_hand_landmarks else np.zeros(21 * 3))
    return np.concatenate([lh, rh])


def preprocess_frame(results):
    kp = extract_keypoints(results).reshape(1, -1)
    return normalize_keypoints(kp)[0]      # same function used in training


In [ ]:

def prob_viz(res, actions, frame, colors):
    out = frame.copy()
    h, w, _ = out.shape
    left_count, row_h, bar_w, top = 13, 30, 120, 40
    for num, prob in enumerate(res):
        c = colors[num % len(colors)]
        if num < left_count:
            y = top + num * row_h
            cv2.rectangle(out, (0, y), (int(prob * 100), y + 20), c, -1)
            cv2.putText(out, actions[num], (0, y + 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        else:
            y = top + (num - left_count) * row_h
            x = w - bar_w
            start = x + bar_w - int(prob * 100)
            cv2.rectangle(out, (start, y), (x + bar_w, y + 20), c, -1)
            cv2.putText(out, actions[num], (start - 20, y + 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out


# One colour per letter for the probability bars
rng = np.random.default_rng(7)
COLORS = [tuple(int(v) for v in rng.integers(40, 255, 3)) for _ in ACTIONS]


In [ ]:

def run_realtime(threshold=0.7, agree_frames=10, max_sentence=5):
    model = tf.keras.models.load_model(MODEL_PATH)

    sequence, sentence, predictions = [], [], []
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Couldn't open the webcam.")
        return

    with mp_holistic.Holistic(min_detection_confidence=0.5,
                              min_tracking_confidence=0.5,
                              model_complexity=0) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image, results = mediapipe_detection(frame, holistic)
            draw_styled_landmarks(image, results)

            sequence.append(preprocess_frame(results))
            sequence = sequence[-SEQUENCE_LENGTH:]

            if len(sequence) == SEQUENCE_LENGTH:
                res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
                top = np.argmax(res)
                predictions.append(top)

                # only commit a letter if the recent predictions agree
                if len(predictions) >= agree_frames and np.unique(predictions[-agree_frames:])[0] == top:
                    if res[top] > threshold:
                        if not sentence or ACTIONS[top] != sentence[-1]:
                            sentence.append(ACTIONS[top])
                    sentence = sentence[-max_sentence:]

                image = prob_viz(res, ACTIONS, image, COLORS)

            cv2.rectangle(image, (0, 0), (image.shape[1], 40), (245, 117, 16), -1)
            cv2.putText(image, ' '.join(sentence), (3, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.imshow('Auslan fingerspelling', image)
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    print("final:", ' '.join(sentence))


# run_realtime()



### Recording more data

If you want to add your own sequences, this writes them in the same folder layout the loader in
section 2 expects. Set `actions` to whichever letters you're recording. There's a 2 second pause at
the start of each sequence so you have time to get your hands into position.


In [ ]:

def collect_data(actions, data_path=DATA_PATH, no_sequences=NO_SEQUENCES,
                 sequence_length=SEQUENCE_LENGTH, start_index=0):
    for action in actions:
        for seq in range(start_index, start_index + no_sequences):
            os.makedirs(os.path.join(data_path, action, str(seq)), exist_ok=True)

    cap = cv2.VideoCapture(0)
    with mp_holistic.Holistic(min_detection_confidence=0.5,
                              min_tracking_confidence=0.5) as holistic:
        for action in actions:
            for seq in range(start_index, start_index + no_sequences):
                for frame_num in range(sequence_length):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    image, results = mediapipe_detection(frame, holistic)
                    draw_styled_landmarks(image, results)

                    cv2.putText(image, f'{action} - sequence {seq}', (15, 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2, cv2.LINE_AA)
                    if frame_num == 0:
                        cv2.putText(image, 'GET READY', (120, 200),
                                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 4, cv2.LINE_AA)
                        cv2.imshow('Collecting', image)
                        cv2.waitKey(2000)
                    else:
                        cv2.imshow('Collecting', image)

                    # save the full 1662-value holistic vector, same as the original data
                    kp = extract_keypoints(results)
                    full = np.concatenate([np.zeros(1662 - 126), kp])
                    np.save(os.path.join(data_path, action, str(seq), str(frame_num)), full)

                    if cv2.waitKey(10) & 0xFF == ord('q'):
                        cap.release(); cv2.destroyAllWindows(); return
    cap.release()
    cv2.destroyAllWindows()


# collect_data(['A', 'B'], start_index=50)



## 11. Notes and limitations

**The test score is optimistic.** All 1300 sequences were recorded by the same person in the same
room. The split is random over sequences, so the test set is 260 sequences of that same person doing
the same letters in the same conditions. It's not leakage in the frame-level sense (whole sequences
stay on one side of the split), but it doesn't tell you how well this works for someone else. If it
performs noticeably worse when you try it on your own hands, that's expected, not a bug. The proper
fix is recording several people and holding one out entirely for testing.

**50 sequences per letter is small.** The augmentation roughly doubles it, but augmented copies aren't
new information - they don't add hand shapes or signing styles the model has never seen.

**The 30-frame window is fixed.** Every letter gets exactly 30 frames whether it's a static shape held
still or a movement letter. Slow signing can run past the window. Time warping helps a bit but doesn't
fully solve it.

**Both hands need to be visible.** Auslan fingerspelling is two-handed, so if one hand drifts out of
frame that half of the input goes to zeros and predictions get unreliable. Worth checking the
detection rates from section 3 if results are poor.

**Things that would improve it, roughly in order of effort:**
- record a few more people and do a leave-one-person-out test, so the reported accuracy means something
- try a GRU or a 1D CNN over time instead of the LSTM - often similar accuracy, trains faster
- add an explicit "nothing" class so the model can say no letter is being signed, instead of always
  picking one of 26
- feed velocity (frame-to-frame differences) alongside the raw coordinates, which usually helps for
  the movement letters
